In [1]:
import pandas as pd

In [2]:
df = pd.read_excel('../data/dcInbox/dcinbox_export_116_b2.xlsx')
unnamed_cols = df.columns.str.contains('^Unnamed')
df = df.loc[:, ~unnamed_cols].copy()

In [ ]:
"""
Policy Phrase Coverage Analysis
Creates a lexicon and calculates what % of emails contain each phrase
"""

LEXICON = [
    'voting rights',
    'election security',
    'health care',
    'healthcare',
    'gun violence',
    'social security',
    'voting rights act',
    'civil rights',
    'affordable care act',
    'obamacare',
    'public health',
    'child care',
    'mental health',
    'climate change',
    'renewable energy',
    'minimum wage',
    'reproductive rights',
    'immigration reform',
    'criminal justice',
    'police reform',
    'abortion rights',
    'abortion access',
    'abortion care',
    'women\'s rights',
    'covid',
    'covid-19',
    'covid-19 pandemic',
    'covid-19 vaccine',
    'covid-19 vaccination',
    'covid-19 vaccination rate',
]

dem_df = df[df['Party'] == 'Democrat'].copy()
print(f"Democrat emails: {len(dem_df)}")

# Combine subject and body for searching
print("\nCombining subject and body text...")
dem_df['full_text'] = (dem_df['Subject'].fillna('').astype(str) + ' ' + 
                        dem_df['Body'].fillna('').astype(str)).str.lower()

# Count emails containing each phrase
results = []

print("\nCalculating phrase coverage...")
for phrase in LEXICON:
    # Count how many emails contain this phrase
    count = dem_df['full_text'].str.contains(phrase, case=False, regex=False).sum()
    percentage = (count / len(dem_df)) * 100
    
    results.append({
        'phrase': phrase,
        'email_count': count,
        'percentage': percentage
    })
    
    print(f"  {phrase:30s}: {count:5d} emails ({percentage:5.2f}%)")

# Create results dataframe
results_df = pd.DataFrame(results).sort_values('percentage', ascending=False)

# Display summary
print("\n" + "="*70)
print("TOP PHRASES BY EMAIL COVERAGE")
print("="*70)

for i, row in enumerate(results_df.head(20).itertuples(), 1):
    print(f"{i:2d}. {row.phrase:30s}: {row.email_count:5d} emails ({row.percentage:5.2f}%)")


Democrat emails: 14693

Combining subject and body text...

Calculating phrase coverage...
  voting rights                 :   427 emails ( 2.91%)
  election security             :   166 emails ( 1.13%)
  health care                   :  3752 emails (25.54%)
  healthcare                    :  1815 emails (12.35%)
  gun violence                  :   588 emails ( 4.00%)
  social security               :  1748 emails (11.90%)
  voting rights act             :   134 emails ( 0.91%)
  civil rights                  :   668 emails ( 4.55%)
  affordable care act           :   705 emails ( 4.80%)
  obamacare                     :    60 emails ( 0.41%)
  public health                 :  3033 emails (20.64%)
  child care                    :   604 emails ( 4.11%)
  mental health                 :  1012 emails ( 6.89%)
  climate change                :   956 emails ( 6.51%)
  renewable energy              :   161 emails ( 1.10%)
  minimum wage                  :   175 emails ( 1.19%)
  reproductiv

In [5]:
"""
Policy Phrase Coverage Analysis - Theme-Based Version
Creates a thematic lexicon and calculates coverage at both phrase and theme levels
"""

# Group related phrases by policy theme
THEMATIC_LEXICON = {
    'Healthcare': [
        'health care',
        'healthcare',
        'public health',
        'mental health',
        'affordable care act',
        'obamacare',
        'covid',
        'covid-19',
        'covid-19 pandemic',
        'covid-19 vaccine',
        'covid-19 vaccination',
    ],
    'Voting Rights': [
        'voting rights',
        'voting rights act',
        'election security',
    ],
    'Gun Policy': [
        'gun violence',
        'gun safety',
        'gun control',
        'gun reform',
    ],
    'Social Safety Net': [
        'social security',
        'child care',
        'minimum wage',
    ],
    'Civil Rights': [
        'civil rights',
        'women\'s rights',
    ],
    'Reproductive Rights': [
        'reproductive rights',
        'abortion rights',
        'abortion access',
        'abortion care',
    ],
    'Climate & Energy': [
        'climate change',
        'renewable energy',
        'clean energy',
        'green energy',
    ],
    'Immigration': [
        'immigration reform',
        'immigration policy',
        'border security',
    ],
    'Criminal Justice': [
        'criminal justice',
        'police reform',
        'criminal justice reform',
    ],
}

dem_df = df[df['Party'] == 'Democrat'].copy()
print(f"Democrat emails: {len(dem_df)}")

# Combine subject and body
print("\nCombining subject and body text...")
dem_df['full_text'] = (dem_df['Subject'].fillna('').astype(str) + ' ' + 
                        dem_df['Body'].fillna('').astype(str)).str.lower()

# Analyze both individual phrases and themes
phrase_results = []
theme_results = []

print("\nCalculating phrase and theme coverage...")

for theme, phrases in THEMATIC_LEXICON.items():
    print(f"\n{theme}:")
    
    # Track which emails mention this theme (any phrase)
    theme_mask = pd.Series([False] * len(dem_df), index=dem_df.index)
    
    for phrase in phrases:
        # Count emails containing this specific phrase
        phrase_mask = dem_df['full_text'].str.contains(phrase, case=False, regex=False)
        count = phrase_mask.sum()
        percentage = (count / len(dem_df)) * 100
        
        phrase_results.append({
            'theme': theme,
            'phrase': phrase,
            'email_count': count,
            'percentage': percentage
        })
        
        print(f"  {phrase:30s}: {count:5d} emails ({percentage:5.2f}%)")
        
        # Add to theme mask
        theme_mask = theme_mask | phrase_mask
    
    # Calculate theme-level coverage
    theme_count = theme_mask.sum()
    theme_percentage = (theme_count / len(dem_df)) * 100
    
    theme_results.append({
        'theme': theme,
        'email_count': theme_count,
        'percentage': theme_percentage,
        'num_phrases': len(phrases)
    })
    
    print(f"  → TOTAL for {theme}: {theme_count:5d} emails ({theme_percentage:5.2f}%)")

# Create results dataframes
phrase_df = pd.DataFrame(phrase_results).sort_values('percentage', ascending=False)
theme_df = pd.DataFrame(theme_results).sort_values('percentage', ascending=False)

# Display theme-level summary
print("\n" + "="*70)
print("THEME COVERAGE (emails mentioning ANY phrase in theme)")
print("="*70)

for i, row in enumerate(theme_df.itertuples(), 1):
    print(f"{i:2d}. {row.theme:25s}: {row.email_count:5d} emails ({row.percentage:5.2f}%) [{row.num_phrases} phrases]")

# Display top individual phrases
print("\n" + "="*70)
print("TOP INDIVIDUAL PHRASES BY COVERAGE")
print("="*70)

for i, row in enumerate(phrase_df.head(20).itertuples(), 1):
    print(f"{i:2d}. {row.phrase:30s} ({row.theme:20s}): {row.email_count:5d} ({row.percentage:5.2f}%)")

Democrat emails: 14693

Combining subject and body text...

Calculating phrase and theme coverage...

Healthcare:
  health care                   :  3752 emails (25.54%)
  healthcare                    :  1815 emails (12.35%)
  public health                 :  3033 emails (20.64%)
  mental health                 :  1012 emails ( 6.89%)
  affordable care act           :   705 emails ( 4.80%)
  obamacare                     :    60 emails ( 0.41%)
  covid                         :  5773 emails (39.29%)
  covid-19                      :  5648 emails (38.44%)
  covid-19 pandemic             :  1988 emails (13.53%)
  covid-19 vaccine              :   234 emails ( 1.59%)
  covid-19 vaccination          :    11 emails ( 0.07%)
  → TOTAL for Healthcare:  8818 emails (60.01%)

Voting Rights:
  voting rights                 :   427 emails ( 2.91%)
  voting rights act             :   134 emails ( 0.91%)
  election security             :   166 emails ( 1.13%)
  → TOTAL for Voting Rights:   567 ema

In [12]:
"""
Capture Full Phrase Matches with Email Metadata
Creates a detailed dataframe showing which emails contain which phrases,
including the actual matched text and email metadata
"""

import re
from collections import defaultdict

def extract_phrase_contexts(text, phrase, context_chars=50):
    """
    Extract the actual phrase matches with surrounding context
    """
    contexts = []
    # Use case-insensitive search
    pattern = re.compile(re.escape(phrase), re.IGNORECASE)
    
    for match in pattern.finditer(text):
        start = max(0, match.start() - context_chars)
        end = min(len(text), match.end() + context_chars)
        context = text[start:end]
        
        # Highlight the matched phrase
        highlighted = context.replace(match.group(), f"**{match.group()}**")
        contexts.append(highlighted)
    
    return contexts

phrase_matches = []

for theme, phrases in THEMATIC_LEXICON.items():
    for phrase in phrases:
        # Find emails containing this phrase
        phrase_mask = dem_df['full_text'].str.contains(phrase, case=False, regex=False)
        matching_emails = dem_df[phrase_mask]
        
        for idx, email_row in matching_emails.iterrows():
            # Extract contexts for this phrase in this email
            contexts = extract_phrase_contexts(email_row['full_text'], phrase)
            
            for context in contexts:
                phrase_matches.append({
                    'email_index': idx,
                    'theme': theme,
                    'phrase': phrase,
                    'matched_context': context,
                    'subject': email_row['Subject'],
                    'sender': email_row.get('Sender', ''),
                    'recipient': email_row.get('Recipient', ''),
                    'date': email_row.get('Date', ''),
                    'party': email_row['Party'],
                    'full_text_length': len(email_row['full_text'])
                })

# Create the detailed matches dataframe
matches_df = pd.DataFrame(phrase_matches)

# Count matches by theme
theme_counts = matches_df.groupby('theme').agg({
    'phrase': 'count',
    'email_index': 'nunique'
}).rename(columns={'phrase': 'total_matches', 'email_index': 'unique_emails'})



# store metadata about matches
for theme in matches_df['theme'].unique():
    theme_matches = matches_df[matches_df['theme'] == theme].head(2)



In [10]:
theme_matches.head(5)

,email_index,theme,phrase,matched_context,subject,sender,recipient,date,party,full_text_length
92621,82,Criminal Justice,criminal justice,"mmigrants, dreamers, and refugees pass meaning...",SURVEY: Your Priorities for the 117th Congress,,,,Democrat,2873
92622,259,Criminal Justice,criminal justice,th disabilities -- out of classrooms and into ...,Senator Bennet's Weekly Update,,,,Democrat,4408
